# Synthetic Positive Wakeword Generator

Generate synthetic wakeword samples using Kokoro TTS with realistic audio augmentations.

## Setup

In [ ]:
# Setup: Install dependencies in conda environment (run once)
# Make sure you're using the 'wakeword' conda environment
# Run these commands in your terminal:
#
# conda activate wakeword
# python -m pip install -r requirements.txt
# python -m pip install ipykernel
# python -m ipykernel install --user --name wakeword --display-name "Python 3 (wakeword)"
#
# IMPORTANT: Use "python -m pip" instead of just "pip" to ensure you're using
# the conda environment's pip, not the system pip!
#
# Then select the "Python 3 (wakeword)" kernel in Jupyter

In [1]:
import numpy as np
import soundfile as sf
from pathlib import Path
from scipy import signal
from IPython.display import Audio, display
import json
import os


# Import our modules
from src.tts import TTSEngine, KOKORO_SAMPLE_RATE, DEFAULT_VOICES
from src.augment import AugmentationChain
from src.audio_utils import (
    resample, to_pcm16, add_silence_padding, 
    apply_gain, apply_lowpass, apply_highpass, apply_bandpass,
    pitch_shift, time_stretch
)

## Download MIT Room Impulse Responses (REQUIRED)

In [ ]:
# Download room impulse responses collected by MIT (REQUIRED)
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html
# These real-world RIRs provide realistic room acoustics for augmentation

from pathlib import Path

output_dir = "./mit_rirs"

# Check if files already exist
if Path(output_dir).exists():
    num_files = len(list(Path(output_dir).glob("*.wav")))
    if num_files > 0:
        print(f"✓ RIR directory exists with {num_files} files.")
        print("Sample files:", list(Path(output_dir).glob("*.wav"))[:5])
    else:
        print("⚠ mit_rirs directory exists but is empty!")
        print("\nPlease run the download script:")
        print("  python download_mit_rirs.py")
else:
    print("⚠ MIT RIR files not found!")
    print("\nTo download them, run this in your terminal:")
    print("  python download_mit_rirs.py")
    print("\nThis will:")
    print("  - Fix the pyarrow/datasets compatibility issue")
    print("  - Download ~270 real room impulse response files (~500MB)")
    print("  - Take a few minutes to complete")
    print("\nAfter that, rerun this cell to verify the files were downloaded.")

## Configuration

In [ ]:
# === CONFIGURE THESE ===
WAKEWORD = "How do you wanna do this!?"
NUM_SAMPLES = 25
MASTER_SEED = 42
OUTPUT_DIR = Path("output")

# RIR directory (REQUIRED - run the download cell above first)
RIR_DIR = Path("mit_rirs")
if not RIR_DIR.exists():
    raise FileNotFoundError(
        f"RIR directory {RIR_DIR} not found! "
        "Please run the 'Download MIT Room Impulse Responses' cell above first."
    )

# Optional: path to background noise files for augmentation
# If None, synthetic white noise will be used
NOISE_DIR = None  # e.g., Path("noise")

# Target format (openWakeWord requirement)
TARGET_SR = 16000

In [ ]:
# Augmentation configuration with recommended values
# Based on best practices for wakeword/speech detection model training
AUG_CONFIG = {
    "gain": {
        # Volume adjustment in decibels
        # Range: [-6, 3] dB is conservative and preserves speech clarity
        # More aggressive: [-12, 6] dB (may cause clipping/distortion)
        # Purpose: Simulates different microphone distances and recording levels
        "db_range": [-6, 3]
    },
    
    "speed": {
        # Speech tempo/speed multiplier
        # Range: [0.9, 1.1] = ±10% variation (standard for speech augmentation)
        # More aggressive: [0.85, 1.15] = ±15% (may sound unnatural)
        # Purpose: Accounts for natural speaking rate variations
        "range": [0.9, 1.1]
    },
    
    "pitch": {
        # Pitch shift in semitones (musical intervals)
        # Probability: 0.5 = 50% of samples get pitch shifted
        # Semitone range: [-2, 2] = ±2 semitones (safe for speech)
        # More aggressive: [-4, 4] semitones (may sound unnatural)
        # Purpose: Accounts for speaker voice pitch variations (age, gender, etc.)
        "probability": 0.5,
        "semitone_range": [-2, 2]
    },
    
    "silence": {
        # Leading/trailing silence padding in seconds
        # Range: [0.1, 0.5] seconds provides natural variation
        # Purpose: Simulates real-world scenarios where wakewords aren't isolated
        #          Helps model learn to ignore background before/after speech
        "leading_range": [0.1, 0.5],
        "trailing_range": [0.1, 0.5]
    },
    
    "reverb": {
        # Room impulse response (reverberation) application
        # Probability: 0.7 = 70% of samples get reverb (high for robustness)
        # Purpose: Simulates different room acoustics (small room, large hall, etc.)
        #          Critical for real-world deployment where devices are in various rooms
        "probability": 0.7
    },
    
    "noise": {
        # Background noise mixing
        # Probability: 0.8 = 80% of samples get noise (high for robustness)
        # SNR range: [15, 30] dB = clean to moderately noisy (good for wakewords)
        #   - 30 dB: Very clean, noise barely audible
        #   - 20 dB: Moderate noise, speech still clear
        #   - 15 dB: Noticeable noise, speech still intelligible
        #   - 10 dB: Heavy noise (may be too aggressive for wakewords)
        # Purpose: Simulates real-world background noise (fans, traffic, TV, etc.)
        "probability": 0.8,
        "snr_db_range": [15, 30]
    },
    
    "lowpass": {
        # Low-pass filter for distance simulation
        # Probability: 0.3 = 30% of samples get filtered
        # Cutoff range: [3000, 6000] Hz simulates moderate distance
        #   - 6000 Hz: Slight muffling (moderate distance)
        #   - 4000 Hz: Noticeable muffling (far distance)
        #   - 3000 Hz: Strong muffling (very far, like phone call quality)
        #   - 2000 Hz: Very aggressive (may be too much)
        # Purpose: Simulates distance from microphone (farther = more high-frequency loss)
        "probability": 0.3,
        "cutoff_range": [3000, 6000]
    },
    
    "bandpass": {
        # Band-pass filter for telephone/radio effect (optional)
        # Probability: 0.1 = 10% of samples (rare, special case)
        # Purpose: Simulates degraded audio channels (phone calls, radio, etc.)
        # Note: This is typically lower probability as it's a special case
        "probability": 0.1,
        "low_hz": 300,   # Telephone band: 300-3400 Hz
        "high_hz": 3400
    }
}

## Initialize TTS Engine

In [ ]:
print("Initializing Kokoro TTS engine...")
tts = TTSEngine()
print(f"Available voices: {DEFAULT_VOICES}")

---
## Audio Manipulation Demos

Interactive examples of each augmentation type.

### 1. Basic TTS Synthesis

In [ ]:
# Generate a single sample
audio_raw = tts.synthesize(WAKEWORD, voice="af_bella", speed=1.0)
print(f"Generated {len(audio_raw)} samples at {KOKORO_SAMPLE_RATE}Hz ({len(audio_raw)/KOKORO_SAMPLE_RATE:.2f}s)")

display(Audio(audio_raw, rate=KOKORO_SAMPLE_RATE))

### 2. Voice Blending

In [ ]:
# Blend two voices (creates unique speaker profiles)
blended_voice = tts.blend_voices("af_bella", "am_michael", ratio=0.5)
audio_blended = tts.synthesize(WAKEWORD, voice=blended_voice, speed=1.0)

print("Blended voice (50% af_bella + 50% am_michael):")
display(Audio(audio_blended, rate=KOKORO_SAMPLE_RATE))

### 3. Speed Variation

In [ ]:
# Speed variations
for speed in [0.85, 1.0, 1.15]:
    audio = tts.synthesize(WAKEWORD, voice="af_bella", speed=speed)
    print(f"Speed {speed}x:")
    display(Audio(audio, rate=KOKORO_SAMPLE_RATE))

### 4. Gain Adjustment

In [ ]:
# Gain adjustment (volume)
# NOTE: IPython Audio widget normalizes playback, so gain differences may not be audible.
# However, the actual audio data IS modified (check np.max(np.abs(audio)) to verify).
audio_base = tts.synthesize(WAKEWORD, voice="af_bella", speed=1.0)

for gain_db in [-6, 0, 3]:
    audio_gained = apply_gain(audio_base.copy(), gain_db)
    peak = np.max(np.abs(audio_gained))
    print(f"Gain {gain_db:+d} dB (peak amplitude: {peak:.4f}):")
    display(Audio(audio_gained, rate=KOKORO_SAMPLE_RATE))

### 5. Low-Pass Filter (Distance Simulation)

In [ ]:
# Low-pass filter simulates distance (muffled sound)
audio_base = tts.synthesize(WAKEWORD, voice="af_bella", speed=1.0)

print("Original:")
display(Audio(audio_base, rate=KOKORO_SAMPLE_RATE))

for cutoff in [6000, 4000, 2000]:
    audio_filtered = apply_lowpass(audio_base.copy(), KOKORO_SAMPLE_RATE, cutoff)
    print(f"Low-pass {cutoff}Hz:")
    display(Audio(audio_filtered, rate=KOKORO_SAMPLE_RATE))

### 6. High-Pass Filter

In [ ]:
# High-pass filter removes low frequencies (tinny sound)
audio_base = tts.synthesize(WAKEWORD, voice="af_bella", speed=1.0)

print("Original:")
display(Audio(audio_base, rate=KOKORO_SAMPLE_RATE))

for cutoff in [100, 300, 500]:
    audio_filtered = apply_highpass(audio_base.copy(), KOKORO_SAMPLE_RATE, cutoff)
    print(f"High-pass {cutoff}Hz:")
    display(Audio(audio_filtered, rate=KOKORO_SAMPLE_RATE))

### 7. Band-Pass Filter (Telephone/Radio Effect)

In [ ]:
# Band-pass filter simulates telephone or radio transmission
audio_base = tts.synthesize(WAKEWORD, voice="af_bella", speed=1.0)

print("Original:")
display(Audio(audio_base, rate=KOKORO_SAMPLE_RATE))

# Telephone band (300-3400 Hz)
audio_phone = apply_bandpass(audio_base.copy(), KOKORO_SAMPLE_RATE, 300, 3400)
print("Telephone (300-3400Hz):")
display(Audio(audio_phone, rate=KOKORO_SAMPLE_RATE))

# AM Radio band (100-5000 Hz)
audio_radio = apply_bandpass(audio_base.copy(), KOKORO_SAMPLE_RATE, 100, 5000)
print("AM Radio (100-5000Hz):")
display(Audio(audio_radio, rate=KOKORO_SAMPLE_RATE))

### 8. Pitch Shifting

In [ ]:
# Pitch shift in semitones (preserves duration)
audio_base = tts.synthesize(WAKEWORD, voice="af_bella", speed=1.0)

print("Original:")
display(Audio(audio_base, rate=KOKORO_SAMPLE_RATE))

for semitones in [-3, -1, 1, 3]:
    audio_pitched = pitch_shift(audio_base.copy(), KOKORO_SAMPLE_RATE, semitones)
    print(f"Pitch {semitones:+d} semitones:")
    display(Audio(audio_pitched, rate=KOKORO_SAMPLE_RATE))

### 9. Time Stretching

In [ ]:
# Time stretch (changes duration, preserves pitch)
audio_base = tts.synthesize(WAKEWORD, voice="af_bella", speed=1.0)

print(f"Original ({len(audio_base)/KOKORO_SAMPLE_RATE:.2f}s):")
display(Audio(audio_base, rate=KOKORO_SAMPLE_RATE))

for factor in [0.8, 1.2, 1.5]:
    audio_stretched = time_stretch(audio_base.copy(), factor)
    print(f"Time stretch {factor}x ({len(audio_stretched)/KOKORO_SAMPLE_RATE:.2f}s):")
    display(Audio(audio_stretched, rate=KOKORO_SAMPLE_RATE))

### 10. Room Impulse Response (Reverb/Convolution)

In [ ]:
def apply_reverb(audio: np.ndarray, rir: np.ndarray) -> np.ndarray:
    """Apply room impulse response via convolution."""
    # Normalize RIR
    rir = rir / np.abs(rir).max()
    # Convolve and trim to original length
    return signal.fftconvolve(audio, rir, mode='full')[:len(audio)]

# Demo with synthetic impulse response (exponential decay)
def create_synthetic_rir(sr: int, decay_time: float = 0.3) -> np.ndarray:
    """Create a simple synthetic room impulse response."""
    length = int(sr * decay_time)
    t = np.linspace(0, decay_time, length)
    rir = np.random.randn(length) * np.exp(-5 * t)
    rir[0] = 1.0  # Direct sound
    return rir

audio_base = tts.synthesize(WAKEWORD, voice="af_bella", speed=1.0)

print("Original (dry):")
display(Audio(audio_base, rate=KOKORO_SAMPLE_RATE))

for decay in [0.1, 0.3, 0.6]:
    rir = create_synthetic_rir(KOKORO_SAMPLE_RATE, decay)
    audio_reverb = apply_reverb(audio_base.copy(), rir)
    print(f"Reverb (decay={decay}s):")
    display(Audio(audio_reverb, rate=KOKORO_SAMPLE_RATE))

### 11. Background Noise Mixing

In [ ]:
def mix_noise(audio: np.ndarray, snr_db: float) -> np.ndarray:
    """Add white noise at specified SNR."""
    noise = np.random.randn(len(audio))
    
    # Calculate scaling for target SNR
    audio_power = np.mean(audio ** 2)
    noise_power = np.mean(noise ** 2)
    target_noise_power = audio_power / (10 ** (snr_db / 10))
    scale = np.sqrt(target_noise_power / noise_power)
    
    return audio + noise * scale

audio_base = tts.synthesize(WAKEWORD, voice="af_bella", speed=1.0)

print("Original (clean):")
display(Audio(audio_base, rate=KOKORO_SAMPLE_RATE))

for snr in [30, 20, 10]:
    audio_noisy = mix_noise(audio_base.copy(), snr)
    print(f"SNR {snr} dB:")
    display(Audio(audio_noisy, rate=KOKORO_SAMPLE_RATE))

### 12. Combined Augmentation Chain

In [ ]:
# Full augmentation chain (as used in batch generation)
# NOTE: Some effects may not be audible due to:
# - IPython Audio widget normalizes playback (gain differences)
# - RIR/noise files may not be available (reverb/noise won't apply)
# - Probability checks may skip some effects
augmenter = AugmentationChain(
    config=AUG_CONFIG,
    rir_dir=RIR_DIR,
    noise_dir=NOISE_DIR
)

audio_base = tts.synthesize(WAKEWORD, voice="af_bella", speed=1.0)
rng = np.random.default_rng(42)

print("Original:")
print(f"  Peak amplitude: {np.max(np.abs(audio_base)):.4f}")
display(Audio(audio_base, rate=KOKORO_SAMPLE_RATE))

# Apply full chain
audio_aug, metadata = augmenter.apply(audio_base.copy(), KOKORO_SAMPLE_RATE, rng)

print(f"\nAugmented (metadata: {metadata}):")
print(f"  Peak amplitude: {np.max(np.abs(audio_aug)):.4f}")
display(Audio(audio_aug, rate=KOKORO_SAMPLE_RATE))

---
## Batch Generation

In [ ]:
import re
from tqdm.notebook import tqdm
from src.manifest import ManifestEntry, ManifestWriter

def slugify(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r'[^\w\s-]', '', text)
    text = re.sub(r'[-\s]+', '_', text)
    return text

def generate_filename(wakeword, voice_config, speed, seed):
    slug = slugify(wakeword)
    if voice_config["voice_type"] == "blend":
        voice_part = f"mix_{voice_config['voice_a'][:2]}_{voice_config['voice_b'][:2]}"
    else:
        voice_part = voice_config["voice"]
    speed_part = f"s{int(speed * 100):03d}"
    return f"{slug}_{voice_part}_{speed_part}_seed{seed}.wav"

In [ ]:
# Setup output
positives_dir = OUTPUT_DIR / "positives"
positives_dir.mkdir(parents=True, exist_ok=True)

manifest = ManifestWriter(OUTPUT_DIR / "manifest.jsonl")
master_rng = np.random.default_rng(MASTER_SEED)

speed_range = AUG_CONFIG["speed"]["range"]
silence_cfg = AUG_CONFIG["silence"]

print(f"Generating {NUM_SAMPLES} samples for '{WAKEWORD}'...")

for i in tqdm(range(NUM_SAMPLES)):
    sample_seed = int(master_rng.integers(0, 2**31))
    rng = np.random.default_rng(sample_seed)
    
    # Random voice
    voice, voice_config = tts.get_random_voice_config(rng)
    
    # Random speed
    speed = rng.uniform(*speed_range)
    
    # Synthesize
    try:
        audio = tts.synthesize(WAKEWORD, voice, speed)
    except Exception as e:
        print(f"Warning: TTS failed for sample {i}: {e}")
        continue
    
    # Augment
    audio, aug_metadata = augmenter.apply(audio, KOKORO_SAMPLE_RATE, rng)
    
    # Add silence padding
    audio = add_silence_padding(
        audio, KOKORO_SAMPLE_RATE,
        silence_cfg["leading_range"],
        silence_cfg["trailing_range"],
        rng
    )
    
    # Resample to 16kHz
    audio = resample(audio, KOKORO_SAMPLE_RATE, TARGET_SR)
    
    # Convert to PCM16
    audio_pcm = to_pcm16(audio)
    
    # Save
    filename = generate_filename(WAKEWORD, voice_config, speed, sample_seed)
    sf.write(positives_dir / filename, audio_pcm, TARGET_SR, subtype='PCM_16')
    
    # Write manifest
    entry = ManifestEntry(
        filename=filename,
        seed=sample_seed,
        wakeword=WAKEWORD,
        voice_config=voice_config,
        speed=round(speed, 3),
        augmentations=aug_metadata
    )
    manifest.write_entry(entry)

print(f"\nDone! Output saved to {OUTPUT_DIR}/")

## Preview Generated Samples

In [ ]:
# Preview a few generated samples
sample_files = list(positives_dir.glob("*.wav"))[:5]

for f in sample_files:
    audio, sr = sf.read(f)
    print(f"{f.name}:")
    display(Audio(audio, rate=sr))